# ⚡ GraphRAG Studio — Autonomous Knowledge Engine

This notebook manages and tests the **GraphRAG Autonomous Engine** (`graphrag_pipeline.py`) and launches the **Streamlit Web Application** (`app.py`).

---
### 🏗️ Architecture & Pipeline Lifecycle
```
  [ Document Ingestion ] ──► (PDF / TXT / Raw Text via PyMuPDF)
            │
            ▼
  [ Token Chunking ]     ──► Semantic Token Splitting (1200 tokens + overlap)
            │
            ▼
  [ LLM Extraction ]     ──► Entity & Relationship Extraction (gpt-4o-mini)
            │
            ▼
  [ Graph Network ]      ──► Deduplication & NetworkX Topology
            │
            ▼
  [ Community Detection] ──► Louvain / Leiden Modularity Clustering
            │
            ▼
  [ Community Reports ]  ──► Hierarchical Summaries & Findings
            │
            ▼
  [ Storage & Visuals ]  ──► Parquet Tables, ChromaDB Chunks, PyVis HTML
```

---
### 🌟 Key Studio Features
1. **⚡ On-Demand Builder**: Ingest any PDF or text file on-demand with live progress streaming.
2. **💬 Multi-Strategy Chatbot**: Global, Local, DRIFT, ChromaDB Vector RAG, and Side-by-Side Comparison.
3. **🕸️ Interactive PyVis Graph**: Full physics simulation with draggable nodes and community clusters.
4. **📊 Parquet & ChromaDB Explorers**: Live searchable tables and vector distance testing.

## 1. System & Database Health Check
Verify OpenAI API connection, GraphRAG parquet datasets, and ChromaDB vector store.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import truststore
truststore.inject_into_ssl()
from dotenv import load_dotenv
import chromadb

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY', '')
print(f"🔑 OpenAI API Key: {'Connected (' + api_key[:7] + '...)' if api_key else 'Missing'}")

# Check Parquet Files
print("\n📊 GraphRAG Parquet Datasets:")
for name in ['entities', 'relationships', 'nodes', 'community_reports']:
    p = Path(f'./ragtest/output/create_final_{name}.parquet')
    if p.exists():
        df = pd.read_parquet(p)
        print(f"  - create_final_{name}.parquet: {len(df)} rows")
    else:
        print(f"  ❌ Missing {p}")

# Check ChromaDB
chroma_client = chromadb.PersistentClient(path='./notebook/chromadb')
coll = chroma_client.get_collection('paper_collection')
print(f"\n🗄️ ChromaDB 'paper_collection': {coll.count()} vector chunks")

## 2. Programmatic On-Demand GraphRAG Builder Test
Test running the modular `GraphRAGEngine` on a sample text string with real-time progress callbacks.

In [ ]:
from graphrag_pipeline import GraphRAGEngine

sample_text = """
Large Language Models (LLMs) have transformed artificial intelligence.
Supervised Fine-Tuning (SFT) trains models on curated instruction datasets.
Parameter-Efficient Fine-Tuning (PEFT) and Low-Rank Adaptation (LoRA) minimize memory consumption by freezing base model weights.
Direct Preference Optimization (DPO) aligns models directly to human preferences without requiring complex reinforcement learning.
"""

engine = GraphRAGEngine(model_name="gpt-4o-mini", temperature=0.0)

def progress_tracker(pct, message, stats):
    print(f"[{int(pct * 100):3d}%] {message}")

print("🚀 Testing On-Demand Graph Builder Pipeline...")
results = engine.build_from_text(sample_text, max_chunks=2, chunk_size=400, progress_callback=progress_tracker)

print(f"\n🎉 Build Complete in {results['elapsed_seconds']}s!")
print(f"- Entities: {results['entities_count']}")
print(f"- Relationships: {results['relationships_count']}")
print(f"- Communities: {results['communities_count']}")

## 3. Launch the Streamlit Studio Web Application
Launch the modernized startup-style web UI on port 8501.

In [ ]:
import sys
import time
import subprocess
import urllib.request
from IPython.display import display, HTML

PORT = 8501
URL = f"http://localhost:{PORT}"

def is_running(url):
    try:
        with urllib.request.urlopen(f"{url}/_stcore/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if is_running(URL):
    print(f"✅ Streamlit Studio is LIVE at {URL}")
else:
    print(f"🚀 Starting Streamlit Studio on port {PORT}...")
    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    for _ in range(10):
        time.sleep(1)
        if is_running(URL):
            print("🎉 Streamlit Studio is now LIVE!")
            break

display(HTML(f'''
<div style="background: #11131A; padding: 24px; border-radius: 12px; border: 1px solid rgba(99, 102, 241, 0.35); margin: 15px 0;">
    <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 8px;">
        <span style="background: #6366F1; color: white; padding: 4px 8px; border-radius: 6px; font-weight: bold; font-size: 0.8rem;">LIVE APP</span>
        <h3 style="color: white; margin: 0; font-size: 1.25rem;">⚡ GraphRAG Studio</h3>
    </div>
    <p style="color: #94A3B8; font-size: 0.95rem; margin-bottom: 16px;">The autonomous on-demand knowledge builder and hybrid search UI is ready.</p>
    <a href="{URL}" target="_blank" style="display: inline-block; background: linear-gradient(135deg, #6366F1, #06B6D4); color: white; padding: 10px 22px; border-radius: 8px; font-weight: 600; text-decoration: none; box-shadow: 0 0 15px rgba(99, 102, 241, 0.4);">👉 Open GraphRAG Studio ({URL})</a>
</div>
'''))

## 4. Process Controller
Stop or restart background Streamlit processes.

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'streamlit run app.py'], capture_output=True, text=True)
print('🛑 Stopped running Streamlit instances. Re-run Cell 3 to restart.')